In [1]:
import random
import numpy as np

# ==============================
# 激活函数：Sigmoid
# ==============================
def sigmoid(x):
    return 1.0 / (1 + np.exp(-x))

# ==============================
# 神经元节点
# ==============================
class Node:
    def __init__(self, layer_idx, node_idx):
        self.layer_idx = layer_idx
        self.node_idx = node_idx
        self.upstream = []
        self.downstream = []
        self.output = 0.0
        self.delta = 0.0

    def set_output(self, value):
        self.output = value

    def calc_output(self):
        total = sum([conn.upstream_node.output * conn.weight for conn in self.upstream])
        self.output = sigmoid(total)

    def calc_output_delta(self, label):
        self.delta = self.output * (1 - self.output) * (label - self.output)

    def calc_hidden_delta(self):
        total = sum([conn.downstream_node.delta * conn.weight for conn in self.downstream])
        self.delta = self.output * (1 - self.output) * total

# ==============================
# 偏置节点
# ==============================
class ConstNode(Node):
    def __init__(self, layer_idx, node_idx):
        super().__init__(layer_idx, node_idx)
        self.output = 1.0

# ==============================
# 网络层
# ==============================
class Layer:
    def __init__(self, layer_idx, node_count):
        self.layer_idx = layer_idx
        self.nodes = [Node(layer_idx, i) for i in range(node_count)]
        self.nodes.append(ConstNode(layer_idx, node_count))

    def set_input(self, data):
        for i, val in enumerate(data):
            self.nodes[i].set_output(val)

    def forward(self):
        for node in self.nodes[:-1]:
            node.calc_output()

# ==============================
# 连接（权重）
# ==============================
class Connection:
    def __init__(self, upstream_node, downstream_node):
        self.upstream_node = upstream_node
        self.downstream_node = downstream_node
        self.weight = random.uniform(-0.1, 0.1)
        self.gradient = 0.0

    def calc_gradient(self):
        self.gradient = self.downstream_node.delta * self.upstream_node.output

    def update_weight(self, lr):
        self.calc_gradient()
        self.weight += lr * self.gradient

    def get_gradient(self):
        return self.gradient

# ==============================
# 神经网络模型
# ==============================
class Network:
    def __init__(self, layers):
        self.layers = [Layer(i, cnt) for i, cnt in enumerate(layers)]
        self.connections = []
        self._build_connections()

    def _build_connections(self):
        for i in range(len(self.layers) - 1):
            curr = self.layers[i]
            next_ = self.layers[i + 1]
            for up_node in curr.nodes:
                for down_node in next_.nodes[:-1]:
                    conn = Connection(up_node, down_node)
                    up_node.downstream.append(conn)
                    down_node.upstream.append(conn)
                    self.connections.append(conn)

    def predict(self, sample):
        self.layers[0].set_input(sample)
        for layer in self.layers[1:]:
            layer.forward()
        return [node.output for node in self.layers[-1].nodes[:-1]]

    def _calc_delta(self, label):
        for i, node in enumerate(self.layers[-1].nodes[:-1]):
            node.calc_output_delta(label[i])
        for layer in reversed(self.layers[:-1]):
            for node in layer.nodes:
                node.calc_hidden_delta()

    def _update_weights(self, lr):
        for conn in self.connections:
            conn.update_weight(lr)

    def train_one(self, sample, label, lr):
        self.predict(sample)
        self._calc_delta(label)
        self._update_weights(lr)

    def train(self, samples, labels, lr, epochs):
        for epoch in range(epochs):
            for s, l in zip(samples, labels):
                self.train_one(s, l, lr)

    # ==============================
    # 梯度检查专用方法
    # ==============================
    def calc_gradient(self):
        for layer in self.layers[:-1]:
            for node in layer.nodes:
                for conn in node.downstream:
                    conn.calc_gradient()

    def get_gradient(self, label, sample):
        self.predict(sample)
        self._calc_delta(label)
        self.calc_gradient()

# ==============================
# 8位二进制编码工具
# ==============================
class Normalizer:
    def __init__(self):
        self.masks = [1, 2, 4, 8, 16, 32, 64, 128]

    def encode(self, num):
        return [0.9 if num & m else 0.1 for m in self.masks]

    def decode(self, vec):
        bits = [1 if v > 0.5 else 0 for v in vec]
        return sum([b * m for b, m in zip(bits, self.masks)])

# ==============================
# 梯度检查函数（核心新增）
# ==============================
def gradient_check(network, sample_feature, sample_label):
    """
    梯度检查：验证反向传播计算的梯度是否正确
    """
    # 网络误差函数：均方误差
    def network_error(vec1, vec2):
        return 0.5 * sum([(v[0] - v[1]) ** 2 for v in zip(vec1, vec2)])

    # 计算当前样本的梯度
    network.get_gradient(sample_feature, sample_label)

    # 对每个权重进行梯度校验
    for conn in network.connections:
        actual_grad = conn.get_gradient()
        eps = 1e-4

        # 权重+微小值
        conn.weight += eps
        err1 = network_error(network.predict(sample_feature), sample_label)

        # 权重-微小值
        conn.weight -= 2 * eps
        err2 = network_error(network.predict(sample_feature), sample_label)

        # 数值梯度
        expected_grad = (err2 - err1) / (2 * eps)

        # 恢复权重
        conn.weight += eps

        print(f"期望梯度: {expected_grad:.6f} | 实际梯度: {actual_grad:.6f}")

# ==============================
# 生成数据
# ==============================
def make_data():
    norm = Normalizer()
    samples, labels = [], []
    for _ in range(32):
        num = random.randint(0, 255)
        vec = norm.encode(num)
        samples.append(vec)
        labels.append(vec)
    return samples, labels

# ==============================
# 测试准确率
# ==============================
def test_accuracy(net):
    norm = Normalizer()
    correct = 0
    for num in range(256):
        vec = norm.encode(num)
        pred = norm.decode(net.predict(vec))
        if pred == num:
            correct += 1
    print(f"✅ 模型准确率: {correct / 256 * 100:.2f}%")

# ==============================
# 主程序
# ==============================
if __name__ == "__main__":
    net = Network([8, 3, 8])
    
    samples, labels = make_data()
    net.train(samples, labels, lr=0.5, epochs=100)
    
    test_accuracy(net)

    # ==============================
    # 执行梯度检查（新增）
    # ==============================
    print("\n====== 梯度检查开始 ======")
    test_sample = [0.9, 0.1, 0.9, 0.1, 0.9, 0.1, 0.9, 0.1]
    test_label  = [0.9, 0.1, 0.9, 0.1, 0.9, 0.1, 0.9, 0.1]
    gradient_check(net, test_sample, test_label)

✅ 模型准确率: 14.45%

====== 梯度检查开始 ======
期望梯度: -0.023715 | 实际梯度: -0.023715
期望梯度: -0.001032 | 实际梯度: -0.001032
期望梯度: 0.060358 | 实际梯度: 0.060358
期望梯度: -0.002635 | 实际梯度: -0.002635
期望梯度: -0.000115 | 实际梯度: -0.000115
期望梯度: 0.006706 | 实际梯度: 0.006706
期望梯度: -0.023715 | 实际梯度: -0.023715
期望梯度: -0.001032 | 实际梯度: -0.001032
期望梯度: 0.060358 | 实际梯度: 0.060358
期望梯度: -0.002635 | 实际梯度: -0.002635
期望梯度: -0.000115 | 实际梯度: -0.000115
期望梯度: 0.006706 | 实际梯度: 0.006706
期望梯度: -0.023715 | 实际梯度: -0.023715
期望梯度: -0.001032 | 实际梯度: -0.001032
期望梯度: 0.060358 | 实际梯度: 0.060358
期望梯度: -0.002635 | 实际梯度: -0.002635
期望梯度: -0.000115 | 实际梯度: -0.000115
期望梯度: 0.006706 | 实际梯度: 0.006706
期望梯度: -0.023715 | 实际梯度: -0.023715
期望梯度: -0.001032 | 实际梯度: -0.001032
期望梯度: 0.060358 | 实际梯度: 0.060358
期望梯度: -0.002635 | 实际梯度: -0.002635
期望梯度: -0.000115 | 实际梯度: -0.000115
期望梯度: 0.006706 | 实际梯度: 0.006706
期望梯度: -0.026351 | 实际梯度: -0.026351
期望梯度: -0.001147 | 实际梯度: -0.001147
期望梯度: 0.067064 | 实际梯度: 0.067064
期望梯度: 0.003536 | 实际梯度: 0.003536
期望梯度: -0.099580 | 实际梯度: -0.099

In [2]:
# 三层神经网络（向量化实现）
# 输入8 → 隐藏3 → 输出8（自编码器）
# 纯矩阵运算 | 速度快 | 教学标准 | 支持梯度检查

import numpy as np

# ==============================
# 激活函数 Sigmoid（向量化）
# ==============================
def sigmoid(x):
    return 1.0 / (1 + np.exp(-x))

# ==============================
# 神经网络：全向量化实现
# ==============================
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size):
        # 权重矩阵（向量化核心）
        self.W1 = np.random.uniform(-0.1, 0.1, (input_size + 1, hidden_size))  # +1 偏置
        self.W2 = np.random.uniform(-0.1, 0.1, (hidden_size + 1, output_size))

        # 缓存前向值（反向传播用）
        self.a0 = None
        self.z1 = None
        self.a1 = None
        self.z2 = None
        self.a2 = None

        # 梯度
        self.dW1 = np.zeros_like(self.W1)
        self.dW2 = np.zeros_like(self.W2)

    # ==============================
    # 前向传播（向量化）
    # ==============================
    def predict(self, X):
        X = np.array(X).reshape(1, -1)
        m = X.shape[0]

        # 输入层 + 偏置
        self.a0 = np.hstack([np.ones((m, 1)), X])

        # 隐藏层
        self.z1 = np.dot(self.a0, self.W1)
        self.a1 = sigmoid(self.z1)
        self.a1 = np.hstack([np.ones((m, 1)), self.a1])  # 加偏置

        # 输出层
        self.z2 = np.dot(self.a1, self.W2)
        self.a2 = sigmoid(self.z2)

        return self.a2.flatten()

    # ==============================
    # 反向传播（向量化）
    # ==============================
    def backward(self, X, y):
        y = np.array(y).reshape(1, -1)
        self.predict(X)

        # 输出层误差
        delta2 = (self.a2 - y) * (self.a2 * (1 - self.a2))
        # 隐藏层误差
        delta1 = np.dot(delta2, self.W2.T)[:, 1:] * (self.a1[:, 1:] * (1 - self.a1[:, 1:]))

        # 梯度计算
        self.dW2 = np.dot(self.a1.T, delta2)
        self.dW1 = np.dot(self.a0.T, delta1)

    # ==============================
    # 单样本训练
    # ==============================
    def train_one(self, X, y, lr):
        self.backward(X, y)
        self.W1 -= lr * self.dW1
        self.W2 -= lr * self.dW2

    # ==============================
    # 批量训练
    # ==============================
    def train(self, X_list, y_list, lr, epochs):
        for _ in range(epochs):
            for X, y in zip(X_list, y_list):
                self.train_one(X, y, lr)

# ==============================
# 8位二进制编解码
# ==============================
class Normalizer:
    def __init__(self):
        self.masks = np.array([1,2,4,8,16,32,64,128])

    def encode(self, num):
        return np.where(self.masks & num, 0.9, 0.1).tolist()

    def decode(self, vec):
        bits = np.array(vec) > 0.5
        return int(np.sum(self.masks[bits]))

# ==============================
# 梯度检查（向量化兼容）
# ==============================
def gradient_check(nn, X, y, eps=1e-4):
    y = np.array(y)

    def error_fn(pred):
        return 0.5 * np.sum((pred - y) ** 2)

    # 检查 W1
    print("======= 检查 W1 梯度 =======")
    for i in range(nn.W1.shape[0]):
        for j in range(nn.W1.shape[1]):
            nn.W1[i,j] += eps
            e1 = error_fn(nn.predict(X))
            nn.W1[i,j] -= 2*eps
            e2 = error_fn(nn.predict(X))
            nn.W1[i,j] += eps

            grad = (e2 - e1) / (2*eps)
            nn.backward(X, y)
            print(f"数值梯度: {grad:.6f} | 反向梯度: {nn.dW1[i,j]:.6f}")

    # 检查 W2
    print("======= 检查 W2 梯度 =======")
    for i in range(nn.W2.shape[0]):
        for j in range(nn.W2.shape[1]):
            nn.W2[i,j] += eps
            e1 = error_fn(nn.predict(X))
            nn.W2[i,j] -= 2*eps
            e2 = error_fn(nn.predict(X))
            nn.W2[i,j] += eps

            grad = (e2 - e1) / (2*eps)
            nn.backward(X, y)
            print(f"数值梯度: {grad:.6f} | 反向梯度: {nn.dW2[i,j]:.6f}")

# ==============================
# 数据集
# ==============================
def make_data():
    norm = Normalizer()
    Xs, ys = [], []
    for _ in range(32):
        n = np.random.randint(0, 256)
        vec = norm.encode(n)
        Xs.append(vec)
        ys.append(vec)
    return Xs, ys

# ==============================
# 准确率测试
# ==============================
def test_accuracy(nn):
    norm = Normalizer()
    correct = 0
    for n in range(256):
        vec = norm.encode(n)
        pred = norm.decode(nn.predict(vec))
        if pred == n:
            correct +=1
    print(f"✅ 准确率: {correct/256*100:.2f}%")

# ==============================
# 主程序
# ==============================
if __name__ == "__main__":
    # 8 → 3 → 8 自编码器
    nn = NeuralNetwork(8, 3, 8)

    # 训练
    Xs, ys = make_data()
    nn.train(Xs, ys, lr=0.5, epochs=100)

    # 测试
    test_accuracy(nn)

    # 梯度检查
    print("\n====== 梯度检查开始 ======")
    test_x = [0.9,0.1,0.9,0.1,0.9,0.1,0.9,0.1]
    test_y = [0.9,0.1,0.9,0.1,0.9,0.1,0.9,0.1]
    gradient_check(nn, test_x, test_y)

✅ 准确率: 10.16%

====== 梯度检查开始 ======
======= 检查 W1 梯度 =======
数值梯度: 0.011092 | 反向梯度: -0.011092
数值梯度: -0.001256 | 反向梯度: 0.001256
数值梯度: 0.015897 | 反向梯度: -0.015897
数值梯度: 0.009982 | 反向梯度: -0.009982
数值梯度: -0.001131 | 反向梯度: 0.001131
数值梯度: 0.014307 | 反向梯度: -0.014307
数值梯度: 0.001109 | 反向梯度: -0.001109
数值梯度: -0.000126 | 反向梯度: 0.000126
数值梯度: 0.001590 | 反向梯度: -0.001590
数值梯度: 0.009982 | 反向梯度: -0.009982
数值梯度: -0.001131 | 反向梯度: 0.001131
数值梯度: 0.014307 | 反向梯度: -0.014307
数值梯度: 0.001109 | 反向梯度: -0.001109
数值梯度: -0.000126 | 反向梯度: 0.000126
数值梯度: 0.001590 | 反向梯度: -0.001590
数值梯度: 0.009982 | 反向梯度: -0.009982
数值梯度: -0.001131 | 反向梯度: 0.001131
数值梯度: 0.014307 | 反向梯度: -0.014307
数值梯度: 0.001109 | 反向梯度: -0.001109
数值梯度: -0.000126 | 反向梯度: 0.000126
数值梯度: 0.001590 | 反向梯度: -0.001590
数值梯度: 0.009982 | 反向梯度: -0.009982
数值梯度: -0.001131 | 反向梯度: 0.001131
数值梯度: 0.014307 | 反向梯度: -0.014307
数值梯度: 0.001109 | 反向梯度: -0.001109
数值梯度: -0.000126 | 反向梯度: 0.000126
数值梯度: 0.001590 | 反向梯度: -0.001590
======= 检查 W2 梯度 =======
数值梯度: 0.124926 | 反向梯度: 

✅ 模型准确率: 10.55%


/var/folders/8x/5qpq8t9100n9cdfszm7nc07h0000gn/T/ipykernel_7769/3340427353.py:32: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  total = sum(conn.upstream_node.output * conn.weight for conn in self.upstream)
/var/folders/8x/5qpq8t9100n9cdfszm7nc07h0000gn/T/ipykernel_7769/3340427353.py:41: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  total = sum(conn.downstream_node.delta * conn.weight for conn in self.downstream)
/var/folders/8x/5qpq8t9100n9cdfszm7nc07h0000gn/T/ipykernel_7769/3340427353.py:155: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  return sum(b * m for b, m in zip(bits, self.masks